<a href="https://colab.research.google.com/github/divya-dataengineer/data-engineering-pipeline-project/blob/main/etl_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from google.colab import files
import pandas as pd

uploaded = files.upload()

#create spark session
spark = SparkSession.builder.appName("ETL Pipeline").getOrCreate()

#upload excel file from local
df = spark.read.csv(r"tips_data.xlsx", header=True, inferSchema=True)

#conver excel to csv
df = pd.read_excel(r"tips_data.xlsx")
df.to_csv("tips.csv", index=False)

#load dataset
#url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
df = spark.read.csv("tips.csv", header=True, inferSchema=True)

#show data
df.show()

#data cleaning
df = df.dropna()
df = df.withColumnRenamed("total_bill", "total_amount")

#transformation
df = df.withColumn("tip_percentage", (col("tip") / col("total_amount")) * 100)

#filtering
df_filtered = df.filter(col("total_amount") > 10)

#aggregation
df_grouped = df.groupBy("day").avg("total_amount", "tip")
df_grouped.show()

#save output
df_filtered.write.csv("output_data", header=True)